# 04 Monte Carlo OOS Risk Dashboard

Use the out-of-sample cluster series produced by notebook 03 to estimate sequence risk, drawdown percentiles, and risk of ruin.

In [ ]:
from pathlib import Path
import sys

def find_project_root():
    candidates = [Path.cwd().resolve(), Path('Z:/SEN05_Autotrading'), Path('//10.11.12.6/Share/SEN05_Autotrading')]
    for candidate in candidates:
        current = candidate
        while True:
            if (current / 'pyproject.toml').exists() and (current / 'backtest_optimize').exists():
                return current
            if current.parent == current:
                break
            current = current.parent
    raise RuntimeError('Could not find project root.')

project_root = find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
BACKTEST_ROOT = project_root / 'backtest_optimize'
SNAPSHOT_DIR = BACKTEST_ROOT / 'outputs' / 'version_snapshots'
OUTPUT_DIR = BACKTEST_ROOT / 'outputs' / 'monte_carlo_runs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import pandas as pd
from IPython.display import Markdown, display

from backtest_optimize.analysis.monte_carlo import eligible_r_values, simulate_paths, summarize_paths
from backtest_optimize.analysis.versioning import load_snapshot, make_run_id, save_snapshot

pd.set_option('display.max_columns', 100)

In [ ]:
# Parent run selection. Set a specific walk-forward snapshot, or use latest.
WALKFORWARD_SNAPSHOT = None
N_RUNS = 2000
RISK_PER_CLUSTER = 0.01
DRAWDOWN_THRESHOLD = 0.10
BLOCK_SIZE = 5
SEED = 42

if WALKFORWARD_SNAPSHOT is None:
    parent_payload = None
    parent_path = None
    for path in sorted(SNAPSHOT_DIR.glob('*.json'), key=lambda item: item.stat().st_mtime, reverse=True):
        payload = load_snapshot(path)
        if payload.get('run_type') == 'walkforward' or str(payload.get('name', '')).startswith('walkforward_'):
            parent_payload = payload
            parent_path = path
            break
    if parent_payload is None:
        raise FileNotFoundError('No walk-forward snapshot found. Run notebook 03 first.')
else:
    parent_path = Path(WALKFORWARD_SNAPSHOT)
    parent_payload = load_snapshot(parent_path)

PARENT_RUN_ID = parent_payload.get('run_id') or parent_payload.get('name')
CLUSTER_FILE = Path(parent_payload['outputs']['oos_clusters'])
clusters = pd.read_csv(CLUSTER_FILE)
r_values = eligible_r_values(clusters)
if len(r_values) == 0:
    raise ValueError('Selected OOS cluster file has no eligible R results.')

display(Markdown('### Parent Walk-Forward Run'))
display(pd.DataFrame([
    {'field': 'snapshot', 'value': str(parent_path)},
    {'field': 'parent_run_id', 'value': PARENT_RUN_ID},
    {'field': 'cluster_file', 'value': str(CLUSTER_FILE)},
    {'field': 'eligible_r_values', 'value': len(r_values)},
]))

In [ ]:
methods = {
    'shuffle': {'method': 'shuffle'},
    'bootstrap': {'method': 'bootstrap'},
    'block_bootstrap': {'method': 'block_bootstrap', 'block_size': BLOCK_SIZE},
}
rows = []
for name, method_kwargs in methods.items():
    paths = simulate_paths(r_values, n_runs=N_RUNS, seed=SEED, **method_kwargs)
    rows.append({
        'method': name,
        **summarize_paths(
            paths, risk_per_cluster=RISK_PER_CLUSTER,
            drawdown_threshold=DRAWDOWN_THRESHOLD,
        ),
    })
summary = pd.DataFrame(rows)
display(Markdown('## Monte Carlo Risk Summary'))
display(summary)

In [ ]:
parent_config = parent_payload.get('config', {})
symbol = parent_config.get('symbol')
timeframe = parent_config.get('timeframe')
RUN_ID = make_run_id('monte_carlo', symbol=symbol, timeframe=timeframe)
summary_path = OUTPUT_DIR / (RUN_ID + '_summary.csv')
summary.to_csv(summary_path, index=False)

snapshot_path = save_snapshot(
    name=RUN_ID, run_id=RUN_ID, run_type='monte_carlo',
    parent_run_id=PARENT_RUN_ID,
    config={
        'source_clusters': CLUSTER_FILE,
        'n_runs': N_RUNS, 'risk_per_cluster': RISK_PER_CLUSTER,
        'drawdown_threshold': DRAWDOWN_THRESHOLD,
        'block_size': BLOCK_SIZE, 'seed': SEED,
    },
    result_summary={
        'eligible_r_values': len(r_values),
        'methods': summary.to_dict('records'),
    },
    outputs={'summary': summary_path},
    signal_file=parent_payload.get('signal_file'),
    market_data_source_id=parent_payload.get('market_data_source_id'),
    repo_root=project_root,
)
print('run_id:', RUN_ID)
print('parent_run_id:', PARENT_RUN_ID)
print(summary_path)
print(snapshot_path)